Run: docker compose -f graph-docker-compose.yml up -d

In [1]:
from neo4j import GraphDatabase
# from tabulate import tabulate
import pandas as pd

In [2]:
# Połączenie z Neo4j
URI = "bolt://neo4j_nosql_lab:7687"
USERNAME = "neo4j"
PASSWORD = "test1234"  # upewnij się, że to pasuje do graph-docker-compose.yml

In [3]:
def run_cypher(driver, query: str):
    with driver.session() as session:
        result = session.run(query)
        records = list(result)

        if not records:
            print("Brak wyników.")
            return

        df = pd.DataFrame([r.data() for r in records])
        return df


def qexec(query: str):
    driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))

    try:
        df = run_cypher(driver, query)
    finally:
        driver.close()

    return df

In [4]:
def reset(tx):
    tx.run("MATCH (n) DETACH DELETE n")

def load_sample_data(tx):
    tx.run("""
        // USERS
        CREATE (u1:User {name: 'U1'}),
        (u2:User {name: 'U2', active: true}),
        (u3:User {name: 'U3', active: true}),
        (u4:User {name: 'U4', active: true}),
        (u5:User {name: 'U5', active: true}),

        // DEVICES
        (d1:Device {id: 'D1'}),
        (d2:Device {id: 'D2'}),
        (d3:Device {id: 'D3'}),

        // NORMAL
        (u1)-[:USES]->(d1),
        (u2)-[:USES]->(d2),

        // FRAUD CLUSTER
        (u3)-[:USES]->(d3),
        (u4)-[:USES]->(d3),
        (u5)-[:USES]->(d3),

        // TRANSFERS
        (u3)-[:TRANSFER {amount: 1000}]->(u1),
        (u4)-[:TRANSFER {amount: 2000}]->(u2)
    """)

driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))

with driver.session() as session:
    session.execute_write(reset)
    session.execute_write(load_sample_data)

print("Dane zostały załadowane!")

driver.close()


Dane zostały załadowane!


### Ćwiczenie 1: Shared Device Detection
Fraud: wiele kont korzysta z jednego urządzenia.
Znajdź urządzenia używane przez >2 userów.

In [5]:
q_cypher = """
// Cypher
// grupujemy po device
MATCH (d:Device)<-[:USES]-(u:User)
WITH d, COUNT(u) AS users

// filtr fraud
WHERE users > 2

RETURN d.id, users;
"""

q_gds_1 = """
// GDS
// projekcja grafu
CALL gds.graph.project(
  'g1',
  ['User', 'Device'],
  { USES: { orientation: 'UNDIRECTED' } }
)
"""

q_gds_2 = """
// community detection
CALL gds.louvain.stream('g1')
YIELD nodeId, communityId

RETURN
  gds.util.asNode(nodeId).name,
  communityId;
"""

In [13]:
qexec(q_cypher)

,path
0,"[{'name': 'U3', 'active': True}, USES, {'id': ..."


In [14]:
# qexec(q_gds_1)
qexec(q_gds_2)

,gds.util.asNode(nodeId).name,communityId
0,U1,5
1,U2,6
2,U3,7
3,U4,7
4,U5,7
5,NaN,5
6,NaN,6
7,NaN,7


### Ćwiczenie 2: Multi-hop powiązania
Fraud często ukrywa się w powiązaniach pośrednich.
Czy U3 jest powiązany z U2?

*..4 = maksymalnie 4 hop-y

In [15]:
q_cypher = """
// Cypher
// szukamy dowolnej ścieżki do długości 4
MATCH path =
  (u3:User {name: 'U3'})-[*..4]-(u2:User {name: 'U2'})

RETURN path;
"""

In [16]:
qexec(q_cypher)

,path
0,"[{'name': 'U3', 'active': True}, USES, {'id': ..."


### Ćwiczenie 3: Shortest Path (Cypher vs GDS)
Najkrótsza ścieżka = najsilniejsze powiązanie.
Znajdź shortest path między U3 i U2.

In [17]:
q_cypher = """
// Cypher
// shortestPath znajduje tylko 1 ścieżkę
MATCH path = shortestPath(
  (u3:User {name: 'U3'})-[*..5]-(u2:User {name: 'U2'})
)

RETURN path;
"""

q_gds_1 = """
// GDS
// projekcja
CALL gds.graph.drop('sp');
"""

q_gds_2 = """
CALL gds.graph.project(
  'sp',
  ['User', 'Device'],
  {
    USES: { orientation: 'UNDIRECTED' },
    TRANSFER: { orientation: 'UNDIRECTED' }
  }
);
"""

q_gds_3 = """
MATCH (s:User {name: 'U3'})
MATCH (t:User {name: 'U2'})

CALL gds.shortestPath.dijkstra.stream('sp', {
  sourceNode: id(s),
  targetNode: id(t)
})
YIELD nodeIds

RETURN
  [n IN nodeIds | gds.util.asNode(n).name] AS path;
"""

In [ ]:
qexec(q_cypher)

In [ ]:
qexec(q_gds_1)
qexec(q_gds_2)
qexec(q_gds_3)

### Ćwiczenie 34: Shortest Path GDS + Filtrowanie Cypher
Najkrótsza ścieżka = najsilniejsze powiązanie.
Znajdź shortest path między U3 i U2.

In [18]:
q_gds_1 = """
// GDS
// projekcja
CALL gds.graph.drop('sp_filtered');
"""

q_gds_2 = """
CALL gds.graph.project.cypher(
  'sp_filtered',

  // NODE FILTER
  '
  MATCH (u:User)
  WHERE u.active = true
  RETURN id(u) AS id

  UNION

  MATCH (d:Device)
  RETURN id(d) AS id
  ',

  // RELATIONSHIP FILTER
  '
  MATCH (u:User)-[r:TRANSFER]->(v:User)
  WHERE r.amount > 1000
  RETURN id(u) AS source, id(v) AS target

  UNION

  MATCH (u:User)-[:USES]->(d:Device)
  RETURN id(u) AS source, id(d) AS target
  '
);
"""

q_gds_3 = """
MATCH (s:User {name: 'U3'})
MATCH (t:User {name: 'U2'})

CALL gds.shortestPath.dijkstra.stream('sp_filtered', {
  sourceNode: id(s),
  targetNode: id(t)
})
YIELD nodeIds

RETURN
  [n IN nodeIds | gds.util.asNode(n).name] AS path;
"""

In [19]:
qexec(q_gds_1)
qexec(q_gds_2)
qexec(q_gds_3)

ClientError: {neo4j_code: Neo.ClientError.Procedure.ProcedureCallFailed} {message: Failed to invoke procedure `gds.graph.drop`: Caused by: org.neo4j.gds.core.loading.GraphNotFoundException: Graph with name `sp_filtered` does not exist on database `neo4j`. It might exist on another database.} {gql_status: 50N42} {gql_status_description: error: general processing exception - unexpected error. Unexpected error has occurred. See debug log for details.}

### Ćwiczenie 5: Fraud Scoring
System musi ocenić ryzyko.
Policz: fraud_score = 1 / path_length + shared_device_bonus

### Cypger

In [20]:
q_cypher = """
MATCH path = shortestPath(
  (u3:User {name: 'U3'})-[*..5]-(u2:User {name: 'U2'})
)

WITH path, length(path) AS len

// sprawdzamy wspólny device
OPTIONAL MATCH (u3:User {name: 'U3'})-[:USES]->(d)<-[:USES]-(u2:User {name: 'U2'})

WITH path, len,
  CASE WHEN d IS NOT NULL THEN 1 ELSE 0 END AS bonus

RETURN
  path,
  1.0 / len + bonus AS fraud_score;
"""

In [21]:
qexec(q_cypher)

,path,fraud_score
0,"[{'name': 'U3', 'active': True}, USES, {'id': ...",0.333333


### GDS

In [22]:
q_gds_1 = """
CALL gds.graph.project(
  'fraud_graph',
  ['User', 'Device'],
  {
    USES: { orientation: 'UNDIRECTED' },
    TRANSFER: { orientation: 'UNDIRECTED' }
  }
);
"""

q_gds_2 = """
MATCH (s:User {name: 'U3'})
MATCH (t:User {name: 'U2'})

CALL gds.shortestPath.dijkstra.stream('fraud_graph', {
  sourceNode: id(s),
  targetNode: id(t)
})
YIELD nodeIds

WITH nodeIds, size(nodeIds) - 1 AS len

OPTIONAL MATCH (s)-[:USES]->(d)<-[:USES]-(t)

RETURN
  [n IN nodeIds | gds.util.asNode(n).name] AS path,
  1.0 / len + CASE WHEN d IS NOT NULL THEN 1 ELSE 0 END AS fraud_score;
"""

In [24]:
# qexec(q_gds_1)
qexec(q_gds_2)

Received notification from DBMS server: <GqlStatusObject gql_status='01N02', status_description='warn: feature deprecated without replacement. id is deprecated and will be removed without a replacement.', position=<SummaryInputPosition line=6, column=15, offset=127>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 127, 'line': 6, 'column': 15}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\nMATCH (s:User {name: 'U3'})\nMATCH (t:User {name: 'U2'})\n\nCALL gds.shortestPath.dijkstra.stream('fraud_graph', {\n  sourceNode: id(s),\n  targetNode: id(t)\n})\nYIELD nodeIds\n\nWITH nodeIds, size(nodeIds) - 1 AS len\n\nOPTIONAL MATCH (s)-[:USES]->(d)<-[:USES]-(t)\n\nRETURN\n  [n IN nodeIds | gds.util.asNode(n).name] AS path,\n  1.0 / l

,path,fraud_score
0,"[U3, None, U4, U2]",1.333333
1,"[U3, None, U4, U2]",1.333333
2,"[U3, None, U4, U2]",1.333333
3,"[U3, None, U4, U2]",1.333333
4,"[U3, None, U4, U2]",1.333333
5,"[U3, None, U4, U2]",1.333333
